# 4C-SCF QOL improvements
In the previous notebook, we were able to run a few iterations of a 4c-scf in He. However there are some things that must be adressed to be directly extracted, such as the size of the basis, eri tensor cleanup and occupation definition. 

Also we will start moving towards a context class similar to the previously established one in the NR-(CS)SCF code. 

In [1]:
from typing import Union
from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray


from py_mods.src.SCF.linalg import transformation_matrix
from py_mods.src.external.DIRAC_ME import (
    build_4c_one_Fock_from_h5,
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_checkpoint,
    build_uncontracted_basis_from_checkpoint,
    occupation_4c,
    eri_classified,
)

from py_mods.src.SCF.scf_kernels import calc_p_matrix_comp

Taking as a reference the `CSRHFContext` class:

In [2]:
@dataclass
class _primitive_KUSCFContext:
    n_bas: int
    nL: int
    nS: int
    S: NDArray[np.float64]
    T: NDArray[np.complex128]
    V: NDArray[np.float64]
    W: NDArray[np.float64]
    eri_classess: NDArray[np.float64]
    n_electrons: int

    # Optional
    occupation: Union[int, NDArray[np.uint8], None] = None

In [3]:
def genetate_primitive_KUSCFContext_from_checkpoint(
    h5_filename,
) -> _primitive_KUSCFContext:

    S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
    H_nuc = V+W+T
    nuc_charge = get_nuc_charge(h5_filename)

    _, nL, nS = build_uncontracted_basis_from_checkpoint(h5_filename)
    eri = full_eri_from_checkpoint(h5_filename)
    eri = eri_classified(eri, nL)

    occ_det = occupation_4c(nS, nL, 2)

    return _primitive_KUSCFContext(
        n_bas=nL * nS,
        nL=nL,
        nS=nS,
        S=S,
        T=T,
        V=V,
        W=W,
        eri_classess=eri,
        n_electrons=nuc_charge,
        occupation=occ_det,
    )

In [ ]:
He_context = genetate_primitive_KUSCFContext_from_checkpoint("files/He_checkpoint.h5")



_primitive_KUSCFContext(n_bas=225, nL=9, nS=25, S=array([[1.        , 0.55316115, 0.21345793, ..., 0.        , 0.        ,
        0.        ],
       [0.55316115, 1.        , 0.68452017, ..., 0.        , 0.        ,
        0.        ],
       [0.21345793, 0.68452017, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 3.        , 0.        ,
        1.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        3.        ]], shape=(68, 68)), T=array([[ 0.+0.j, -0.-0.j, -0.-0.j, ..., -0.-0.j, -0.-0.j, -0.-0.j],
       [ 0.+0.j,  0.+0.j, -0.-0.j, ..., -0.-0.j, -0.-0.j, -0.-0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j, ..., -0.-0.j, -0.-0.j, -0.-0.j],
       ...,
       [-0.+0.j, -0.+0.j, -0.+0.j, ...,  0.-0.j, -0.+0.j, -0.+0.j],
       [-0.+0.j, -0.+0.j, -0.+0.j, ...,  0.-0.j,  0.-0.j, -0.+0.j],
  